<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/22_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [149]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as plt
import random

random.seed(27)

In [150]:
import random

kisiler = ["Boran", "Deniz", "Ayhan", "Ferda", "Zeynep", "Elif", "Ata", "Yaren", "İbo", "Can", "Merve", "Ali", "Ayşe", "Selim", "Murat", "Selin", "Hakan", "Buse", "Kerem", "Ceren", "Yiğit", "Zeyd"]
zamanlar = ["sabah", "öğle", "akşam", "dün", "bugün", "hafta sonu", "geçen gün", "gece yarısı", "gecenin bir yarısı", "şafak vakti", "ikindi vakti", "geceleyin"]
yer_e = ["parka", "okula", "eve", "kütüphaneye", "denize", "pazara", "bahçeye", "sinemaya", "markete", "ofise", "cafeye", "tiyatroya", "spora", "ormana"]
yer_de = ["parkta", "okulda", "evde", "kütüphanede", "denizde", "pazarda", "bahçede", "sinemada", "cafede", "ofiste", "tiyatroda", "sporda", "ormanda"]
isler = [
    ("kitap", "okudu"),
    ("çay", "içti"),
    ("top", "oynadı"),
    ("resim", "yaptı"),
    ("kahve", "hazırladı"),
    ("yemek", "pişirdi"),
    ("müzik", "dinledi"),
    ("ders", "çalıştı"),
    ("kod", "yazdı"),
    ("proje", "geliştirdi"),
    ("yazı", "kaleme aldı"),
    ("film", "izledi")
]
havalar = ["güzel", "soğuk", "sıcak", "yağmurlu", "güneşli", "rüzgarlı", "bulutlu", "karlı", "sisli", "fırtınalı", "ılık", "serin"]
sifatlar = ["mutlu", "yorgun", "neşeli", "sessiz", "heyecanlı", "huzurlu", "dalgın", "endişeli", "şaşkın", "kararlı", "meraklı", "sakin"]

def cumle_uret():
    sablonlar = [
        lambda: f"{random.choice(zamanlar).capitalize()} {random.choice(kisiler)} {random.choice(yer_de)} {random.choice(sifatlar)} bir şekilde {random.choice(isler)[0]} {random.choice(isler)[1]}",
        lambda: f"Hava {random.choice(havalar)}ken {random.choice(zamanlar)}leyin {random.choice(kisiler)} {random.choice(yer_e)} gitti",
        lambda: f"{random.choice(kisiler)} {random.choice(yer_de)} {random.choice(isler)[0]} {random.choice(isler)[1]}",
        lambda: f"{random.choice(zamanlar).capitalize()} hava çok {random.choice(havalar)}dı bu yüzden {random.choice(kisiler)} biraz {random.choice(sifatlar)} hissediyordu",
        lambda: f"{random.choice(kisiler)} {random.choice(zamanlar)} {random.choice(yer_e)} giderek {random.choice(isler)[0]} {random.choice(isler)[1]}",
        lambda: f"{random.choice(kisiler)} {random.choice(isler)[1]}",
        lambda: f"{random.choice(zamanlar).capitalize()} {random.choice(kisiler)} {random.choice(yer_e)} hızlıca koştu çünkü hava fırtınalıydı",
        lambda: f"{random.choice(kisiler)} {random.choice(sifatlar)} bir şekilde {random.choice(isler)[0]} {random.choice(isler)[1]} ve sonra {random.choice(yer_e)} gitti",
        lambda: f"Hava aniden {random.choice(havalar)} olunca {random.choice(kisiler)} {random.choice(yer_de)} {random.choice(isler)[0]} {random.choice(isler)[1]}"
    ]

    secilen_sablon = random.choice(sablonlar)
    return secilen_sablon()

In [151]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_count = 2

    def add_sentence(self, sentence):
        words = self.tokenize(sentence)
        for word in words:
            self.add_word(word.lower())

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.word_count
            self.idx2word[self.word_count] = word
            self.word_count += 1

    def tokenize(self, sentence):
        tokens = []
        for kelime in sentence.split():
            tokens.append(kelime)
        return tokens

vocab = Vocabulary()

for _ in range(5000):
    cumle = cumle_uret()
    vocab.add_sentence(cumle)

print(f"Sözlük oluşturuldu. Toplam kelime sayısı: {len(vocab.word2idx)}")

Sözlük oluşturuldu. Toplam kelime sayısı: 168


In [152]:
def encoder(sentence, vocabulary):
    tokens = vocabulary.tokenize(sentence)
    kodlanmis = []
    for token in tokens:
        t_low = token.lower()
        if t_low in vocabulary.word2idx:
            kodlanmis.append(vocabulary.word2idx[t_low])
        else:
            sözlük_kelimeleri = [w for w in vocabulary.word2idx.keys() if w not in ["<PAD>", "<UNK>"]]
            if not sözlük_kelimeleri:
                kodlanmis.append(1)
                continue
            def get_ngrams(w, n=4):
                return set(w[i:i+n] for i in range(len(w)-n+1))
            hedef_ngrams = get_ngrams(t_low)
            en_iyi_skor = -1
            en_yakin_kelime = sözlük_kelimeleri[0]
            for w in sözlük_kelimeleri:
                w_ngrams = get_ngrams(w)
                if not hedef_ngrams or not w_ngrams:
                    skor = 0
                else:
                    skor = len(hedef_ngrams.intersection(w_ngrams)) / len(hedef_ngrams.union(w_ngrams))
                if skor > en_iyi_skor:
                    en_iyi_skor = skor
                    en_yakin_kelime = w
            kodlanmis.append(vocabulary.word2idx[en_yakin_kelime])
    return kodlanmis

def decoder(indices, vocabulary):
    decoded_words = [vocabulary.idx2word.get(idx, "<UNK>") for idx in indices]
    return " ".join(decoded_words).capitalize()

In [153]:
cumleler = [cumle_uret() for _ in range(1000)]
kodlanmis_cumleler = [encoder(c, vocab) for c in cumleler]

max_uzunluk = 0
for c in kodlanmis_cumleler:
    uzunluk = len(c)
    if uzunluk > max_uzunluk:
        max_uzunluk = uzunluk

baglam = max_uzunluk

dolgulu_listeler = []
for kc in kodlanmis_cumleler:
    fark = max_uzunluk - len(kc)
    dolgulu = kc + [0] * fark
    dolgulu_listeler.append(dolgulu)

veri_tensoru = torch.tensor(dolgulu_listeler, dtype=torch.long)

toplam_veri = len(veri_tensoru)
egitim_boyutu = int(toplam_veri * 0.85)
test_boyutu = toplam_veri - egitim_boyutu

indeksler = torch.randperm(toplam_veri)

egitim_indeksleri = indeksler[:egitim_boyutu]
test_indeksleri = indeksler[egitim_boyutu:]

egitim_kumesi = veri_tensoru[egitim_indeksleri]
test_kumesi = veri_tensoru[test_indeksleri]
vocab_boyutu = len(vocab.word2idx)

print(f"Toplam üretilen cümle sayısı: {toplam_veri}")
print(f"En uzun cümle boyutu: {max_uzunluk}")
print(f"Eğitim Kümesi Şekli (Shape): {egitim_kumesi.shape}")
print(f"Test Kümesi Şekli (Shape): {test_kumesi.shape}")

Toplam üretilen cümle sayısı: 1000
En uzun cümle boyutu: 12
Eğitim Kümesi Şekli (Shape): torch.Size([850, 12])
Test Kümesi Şekli (Shape): torch.Size([150, 12])


In [154]:
yigin = 32

def yigin_al(bolum):
    d = egitim_kumesi if bolum == "egitim" else test_kumesi
    indeksler = torch.randint(0, len(d), (yigin,))
    x = d[indeksler]
    y = torch.roll(x, shifts=-1, dims=1)
    y[:, -1] = 0
    return x, y

x, y = yigin_al("egitim")
print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: torch.Size([32, 12])
y shape: torch.Size([32, 12])


In [155]:
class MaskeliDikkat(nn.Module):
    def __init__(self, d_model, n_kafa):
        super().__init__()
        self.n_kafa = n_kafa
        self.d_k = d_model // n_kafa
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.cikis = nn.Linear(d_model, d_model)
        self.register_buffer("maske", torch.tril(torch.ones(baglam, baglam)))

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q = q.view(B, L, self.n_kafa, self.d_k).transpose(1, 2)
        k = k.view(B, L, self.n_kafa, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.n_kafa, self.d_k).transpose(1, 2)
        skor = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        skor = skor.masked_fill(self.maske[:L, :L][None, None] == 0, float("-inf"))
        agirlik = F.softmax(skor, dim=-1)
        cikti = (agirlik @ v).transpose(1, 2).reshape(B, L, D)
        return self.cikis(cikti)

In [156]:
class GPTBlok(nn.Module):
    def __init__(self, d_model, n_kafa):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.dikkat = MaskeliDikkat(d_model, n_kafa)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        x = x + self.dikkat(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, d_model=64, n_kafa=4, n_katman=2):
        super().__init__()
        self.kelime_gomme = nn.Embedding(vocab_boyutu, d_model)
        self.konum_gomme = nn.Embedding(baglam, d_model)
        self.blok = nn.Sequential(*[GPTBlok(d_model, n_kafa) for _ in range(n_katman)])
        self.norm = nn.LayerNorm(d_model)
        self.cikis = nn.Linear(d_model, vocab_boyutu)

    def forward(self, x, hedef=None):
        B, L = x.shape
        h = self.kelime_gomme(x) + self.konum_gomme(torch.arange(L, device=x.device))
        h = self.norm(self.blok(h))
        logits = self.cikis(h)
        if hedef is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, vocab_boyutu), hedef.view(-1))
        return logits, loss

    @torch.no_grad()
    def uret(self, baslangic_metni, n_kelime, sicaklik=0.5):
        indeksler = encoder(baslangic_metni, vocab)
        x = torch.tensor(indeksler, dtype=torch.long).unsqueeze(0)
        for _ in range(n_kelime):
            x_kirp = x[:, -baglam:]
            logits = self(x_kirp)
            logits = logits[:, -1, :] / sicaklik
            olasilik = F.softmax(logits, dim=-1)
            sonraki = torch.multinomial(olasilik, num_samples=1)
            if sonraki.item() == 0:
                break
            x = torch.cat((x, sonraki), dim=1)
        return decoder(x[0].tolist(), vocab)

In [157]:
class ErkenDurdurma:
    def __init__(self, sabir_limiti=50, ani_artis_orani=2.0):
        self.sabir_limiti = sabir_limiti
        self.ani_artis_orani = ani_artis_orani
        self.en_iyi_kayip = float('inf')
        self.sayac = 0

    def kontrol_et(self, guncel_kayip, onceki_kayip=None):
        if onceki_kayip is not None and guncel_kayip >= onceki_kayip * self.ani_artis_orani:
            print(f"\n[ERKEN DURDURMA] Ani ve aşırı kayıp artışı tespit edildi (Önceki: {onceki_kayip:.4f} -> Güncel: {guncel_kayip:.4f}). Eğitim durduruluyor!")
            return True

        if guncel_kayip < self.en_iyi_kayip:
            self.en_iyi_kayip = guncel_kayip
            self.sayac = 0
        elif onceki_kayip is not None and guncel_kayip < onceki_kayip:
            self.sayac = 0
        else:
            self.sayac += 1
            print(f"\n[UYARI] Kayıp düşmedi. Sabır Sayacı: {self.sayac}/{self.sabir_limiti}")
            if self.sayac >= self.sabir_limiti:
                print(f"[ERKEN DURDURMA] Kayıp {self.sabir_limiti} adım boyunca iyileşmedi. Eğitim sonlandırılıyor!")
                return True
        return False

In [161]:
import torch.optim as optim

model = MiniGPT(d_model=64, n_kafa=4, n_katman=2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
erken_durdurucu = ErkenDurdurma(sabir_limiti=4, ani_artis_orani=2.0)

adim_sayisi = 500
onceki_val_loss = None

print("Eğitim başlıyor...\n")
for adim in range(1, adim_sayisi + 1):
    model.train()
    x_train, y_train = yigin_al("egitim")
    logits, train_loss = model(x_train, y_train)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        x_val, y_val = yigin_al("test")
        _, val_loss = model(x_val, y_val)
        val_loss_val = val_loss.item()

    if adim % 10 == 0 or adim == 1:
        print(f"Adım {adim:03d} | Eğitim Kaybı: {train_loss.item():.4f} | Doğrulama Kaybı: {val_loss_val:.4f}")

    durmali_mi = erken_durdurucu.kontrol_et(val_loss_val, onceki_val_loss)
    if durmali_mi:
        print(f"\n[BİLGİ] Eğitim {adim}. adımda erken durduruldu!")
        break

    onceki_val_loss = val_loss_val

Eğitim başlıyor...

Adım 001 | Eğitim Kaybı: 5.0563 | Doğrulama Kaybı: 4.7410

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4
Adım 010 | Eğitim Kaybı: 3.5116 | Doğrulama Kaybı: 3.1371

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 2/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4
Adım 020 | Eğitim Kaybı: 3.0038 | Doğrulama Kaybı: 2.5959

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4
Adım 030 | Eğitim Kaybı: 2.8063 | Doğrulama Kaybı: 2.4559

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4
Adım 040 | Eğitim Kaybı: 2.1138 | Doğrulama Kaybı: 2.4593

[UYARI] Kayıp düşmedi. Sabır Sayacı: 2/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4

[UYARI] Kayıp düşmedi. Sabır Sayacı: 1/4
Adım 050 | Eğitim Kaybı: 2.0899 | Doğrulama Kaybı: 2.2682

[

In [162]:
model.eval()
for basla in ["boran", "naber", "selim","Zeyd"]:
    uretilen_metin = model.uret(basla, n_kelime=30, sicaklik=0.7)
    print(f"Başlangıç: {basla} -> Üretilen: {uretilen_metin}")
    print("--" * 40)

Başlangıç: boran -> Üretilen: Boran sonu denize giderek kitap oynadı
--------------------------------------------------------------------------------
Başlangıç: naber -> Üretilen: İbo pişirdi
--------------------------------------------------------------------------------
Başlangıç: selim -> Üretilen: Selim geliştirdi
--------------------------------------------------------------------------------
Başlangıç: Zeyd -> Üretilen: Zeyd bugün bahçeye giderek proje geliştirdi
--------------------------------------------------------------------------------
